# 4K → 1080p converter

**No downloads to your computer.** Runs on Google's machines and works straight on the Drive folder.

1. Menu **Runtime → Change runtime type → T4 GPU → Save** (faster, optional).
2. Press ▶ on block 1 and allow access to your Google account.
3. Paste the Drive folder link into the **FOLDER_LINK** box in block 2 and press ▶. No code to edit.

For each video bigger than 1080p it uploads a 1080p copy next to it and moves the 4K original into a **4K originals** subfolder. Videos already 1080p or smaller are skipped. Then name the files in Shoot Renamer.

---

**ES:** Menú **Runtime → Change runtime type → T4 GPU → Save**. Pulsa ▶ en el bloque 1 y da acceso a tu cuenta de Google. Pega el enlace de la carpeta de Drive en la casilla **FOLDER_LINK** del bloque 2 y pulsa ▶. Cada vídeo más grande que 1080p se convierte: la copia 1080p queda en la carpeta y el original 4K se mueve a la subcarpeta **4K originals**.

In [ ]:
#@title 1️⃣ Press ▶ to sign in to Google Drive / Pulsa ▶ para iniciar sesión
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
drive = build('drive', 'v3')
me = drive.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
import subprocess
gpu = bool(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip())
print('Signed in as', me, '| GPU:', 'yes' if gpu else 'no (slower, still works)')

In [ ]:
#@title ⬇️ Paste the Drive folder link in the box on the right, then press ▶ / Pega el enlace de la carpeta en la casilla y pulsa ▶
FOLDER_LINK = ''  #@param {type:"string"}
INCLUDE_SUBFOLDERS = False  #@param {type:"boolean"}
QUALITY = 18  #@param {type:"slider", min:16, max:26, step:1}
#@markdown How many videos per run. Press ▶ again to do the next batch; finished videos are skipped automatically. / Vídeos por tanda. Pulsa ▶ otra vez para la siguiente; los ya hechos se saltan solos.
BATCH_SIZE = 20  #@param {type:"integer"}
#@markdown Only if you can't edit the folder above (view-only): where to save the 1080p videos. Leave empty to create one in your Drive automatically. / Solo si la carpeta es de solo lectura: dónde guardar los vídeos. Déjalo vacío y se crea una en tu Drive.
OUTPUT_FOLDER_LINK = ''  #@param {type:"string"}

import re, os, json, time, subprocess
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

if not FOLDER_LINK.strip(): raise SystemExit('❗ The link box is empty. Paste the folder link into the FOLDER_LINK box above. / La casilla del enlace está vacía.')
m = re.search(r'folders/([A-Za-z0-9_-]+)', FOLDER_LINK)
root_id = m.group(1) if m else FOLDER_LINK.strip().split('?')[0]
KW = dict(supportsAllDrives=True)
LW = dict(supportsAllDrives=True, includeItemsFromAllDrives=True)
FOLDER_MIME = 'application/vnd.google-apps.folder'

def children(fid):
    out, tok = [], None
    while True:
        r = drive.files().list(q=f"'{fid}' in parents and trashed=false", pageSize=1000, pageToken=tok,
              fields='nextPageToken,files(id,name,mimeType,parents,size,videoMediaMetadata(width,height))', **LW).execute()
        out += r.get('files', []); tok = r.get('nextPageToken')
        if not tok: return out

def can_write(fid):
    return drive.files().get(fileId=fid, fields='capabilities(canAddChildren)', **KW).execute().get('capabilities', {}).get('canAddChildren', False)

def ensure_folder(name, parent):
    q = f"name='{name.replace(chr(39), chr(92) + chr(39))}' and mimeType='{FOLDER_MIME}' and trashed=false and '{parent}' in parents"
    r = drive.files().list(q=q, fields='files(id)', **LW).execute().get('files', [])
    if r: return r[0]['id']
    return drive.files().create(body={'name': name, 'mimeType': FOLDER_MIME, 'parents': [parent]}, fields='id', **KW).execute()['id']

_out = {}
def output_for(parent, path):
    # where the 1080p file goes: next to the original if we can write there, otherwise into a folder we own
    if parent in _out: return _out[parent]
    if can_write(parent):
        _out[parent] = (parent, True)
    else:
        global out_root
        if out_root is None:
            m2 = re.search(r'folders/([A-Za-z0-9_-]+)', OUTPUT_FOLDER_LINK)
            out_root = m2.group(1) if m2 else (OUTPUT_FOLDER_LINK.strip() or ensure_folder('1080p – ' + root_name, 'root'))
        sub = out_root if path == root_name else ensure_folder(path.split('/')[-1], out_root)
        _out[parent] = (sub, False)
    return _out[parent]
out_root = None

def originals_folder(parent):
    for f in children(parent):
        if f['mimeType'] == FOLDER_MIME and f['name'] == '4K originals': return f['id']
    return drive.files().create(body={'name': '4K originals', 'mimeType': FOLDER_MIME, 'parents': [parent]}, fields='id', **KW).execute()['id']

def probe(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
        'stream=width,height:stream_tags=rotate:stream_side_data=rotation', '-of', 'json', path], capture_output=True, text=True)
    s = json.loads(r.stdout)['streams'][0]; w, h = int(s['width']), int(s['height'])
    rot = int(s.get('tags', {}).get('rotate', 0) or 0)
    for sd in s.get('side_data_list', []) or []:
        if 'rotation' in sd: rot = int(sd['rotation'])
    return (h, w) if abs(rot) % 180 == 90 else (w, h)

import google.auth, google.auth.transport.requests
_creds, _ = google.auth.default()
def probe_remote(fid):
    # read only the video header straight from Drive (no full download) to get its size
    try:
        if not _creds.valid: _creds.refresh(google.auth.transport.requests.Request())
        url = f'https://www.googleapis.com/drive/v3/files/{fid}?alt=media&supportsAllDrives=true'
        r = subprocess.run(['ffprobe', '-v', 'error', '-headers', f'Authorization: Bearer {_creds.token}\r\n',
            '-select_streams', 'v:0', '-show_entries', 'stream=width,height:stream_tags=rotate:stream_side_data=rotation',
            '-of', 'json', url], capture_output=True, text=True, timeout=120)
        st = json.loads(r.stdout)['streams'][0]; w, h = int(st['width']), int(st['height'])
        return w, h
    except Exception:
        return None

def new_name(name):
    stem = os.path.splitext(name)[0]
    if stem.startswith('._'): stem = stem[2:]
    m = re.match(r'^(\d{4}_.+?_.+?_.+?_\d{2,3})_(3840|2160|4K|4k|3840px|2160px)$', stem)
    return (m.group(1) if m else stem + '') + '_1080p.mp4'

def convert(src, dst):
    vf = "scale='if(gt(iw,ih),1920,-2)':'if(gt(iw,ih),-2,1920)'"
    venc = ['-c:v', 'h264_nvenc', '-preset', 'p5', '-rc', 'vbr', '-cq', str(QUALITY + 1), '-b:v', '0'] if gpu \
        else ['-c:v', 'libx264', '-preset', 'medium', '-crf', str(QUALITY)]
    return subprocess.run(['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', src, '-vf', vf, *venc,
        '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart', dst], capture_output=True, text=True)

def walk(fid, path):
    for f in children(fid):
        if f['mimeType'] == FOLDER_MIME:
            if INCLUDE_SUBFOLDERS and f['name'] != '4K originals': yield from walk(f['id'], path + '/' + f['name'])
        elif f['mimeType'].startswith('video/') and not (f['name'].startswith('._') and int(f.get('size') or 0) < 1_000_000):
            yield f, fid, path

root_name = drive.files().get(fileId=root_id, fields='name', **KW).execute()['name']
videos = sorted(walk(root_id, root_name), key=lambda v: (v[2], v[0]['name'].lower()))
_names = {}
def names_in(fid):
    if fid not in _names: _names[fid] = {f['name'] for f in children(fid)}
    return _names[fid]
todo = []
already = small = 0
for f, parent, path in videos:
    vm = f.get('videoMediaMetadata') or {}
    target, _ = output_for(parent, path)
    if new_name(f['name']) in names_in(target): already += 1; continue
    if not vm.get('width'):
        print(f'  checking size of {f["name"]}…', end='\r', flush=True)
        dims = probe_remote(f['id'])
        if dims: vm = {'width': dims[0], 'height': dims[1]}
    if vm.get('width') and max(vm['width'], vm['height']) <= 1920: small += 1; continue
    todo.append((f, parent, path))
batch = todo[:max(1, int(BATCH_SIZE))]
print(f'"{root_name}": {len(videos)} video(s) · {small} already 1080p or smaller · {already} converted earlier · {len(todo)} to do')
print(f'This run: {len(batch)} video(s)\n')
os.makedirs('/content/work', exist_ok=True)
done = skipped = failed = 0
for i, (f, parent, path) in enumerate(batch, 1):
    print(f'[{i}/{len(batch)}]', end=' ')
    t = time.time(); src = '/content/work/src' + os.path.splitext(f['name'])[1]; dst = '/content/work/out.mp4'
    try:
        print(f'  > {f["name"]}  downloading…', end=' ', flush=True)
        with open(src, 'wb') as fh:
            dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=f['id'], **KW), chunksize=64 * 1024 * 1024)
            while not dl.next_chunk()[1]: pass
        w, h = probe(src)
        if max(w, h) <= 1920:
            print(f'already {w}x{h}, skipped'); skipped += 1; continue
        print(f'{w}x{h} converting…', end=' ', flush=True)
        r = convert(src, dst)
        if r.returncode != 0: raise RuntimeError(r.stderr[-300:])
        print('uploading…', end=' ', flush=True)
        name = new_name(f['name'])
        target, own_folder = output_for(parent, path)
        drive.files().create(body={'name': name, 'parents': [target]},
            media_body=MediaFileUpload(dst, mimetype='video/mp4', resumable=True, chunksize=64 * 1024 * 1024), fields='id', **KW).execute()
        if own_folder:
            keep = originals_folder(parent)
            drive.files().update(fileId=f['id'], addParents=keep, removeParents=parent, fields='id', **KW).execute()
        print(f'✓ {name} ({time.time() - t:.0f}s)'); done += 1
        names_in(target).add(name)
    except Exception as e:
        print('FAILED:', str(e)[:300]); failed += 1
    finally:
        for p in (src, dst):
            if os.path.exists(p): os.remove(p)
left = len(todo) - done - skipped
print(f'\nConverted {done} · skipped {skipped} · failed {failed}')
if left > 0: print(f'\n▶ {left} video(s) left. Press ▶ on this block again for the next batch. / Quedan {left}: pulsa ▶ otra vez para la siguiente tanda.')
else: print('\n✅ All videos in this folder are done. / Todos los vídeos de esta carpeta están listos.')
if out_root:
    print(f'\n🔒 The folder is view-only, so the 1080p videos were saved in YOUR Drive (originals untouched):')
    print(f'   https://drive.google.com/drive/folders/{out_root}')
    print('   Load that link in Shoot Renamer to name them. / Carga ese enlace en Shoot Renamer para ponerles nombre.')
elif done: print('Now name them in Shoot Renamer: https://jordialbanell.github.io/shoot-renamer/')